In [61]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

import utils



In [62]:
all_ccf_df = pd.read_csv('all_ccf_df.csv')

In [63]:
all_ccf_df.columns

Index(['star_from_file', 'rjd', 'vrad', 'svrad', 'fwhm', 'sig_fwhm',
       'bis_span', 'sig_bis_span', 'contrast', 'sig_contrast', 's_mw', 'sig_s',
       'ha', 'sig_ha', 'na', 'sig_na', 'ca', 'sig_ca', 'rhk', 'sig_rhk',
       'berv', 'weight', 'file_rootpath', 'observation_file', 'true_vrad1',
       'true_vrad2', 'observation_name', 'ccf_fits_path', 'ccf_file_name',
       'iccf_bis', 'iccf_bis_error', 'iccf_fwhm', 'iccf_fwhm_error',
       'iccf_object', 'iccf_rv', 'iccf_rv_error', 'iccf_vspan', 'iccf_wspan',
       'iccf_ccf', 'iccf_rv_grid', 'iccf_contrast', 'iccf_contrast_error',
       'star', 'Name', 'gaia_dr3', 'Teff', 'eTeff', 'Logg', 'eLogg', '[Fe/H]',
       'e[Fe/H]', 'Vt', 'eVt', 'Gmag', 'Plx', 'Distance', 'Mass_t', 'Radius_t',
       'SWFlag', 'Reference', 'Link', 'Update', 'Database'],
      dtype='object')

In [64]:
all_ccf_df.head(2)

,star_from_file,rjd,vrad,svrad,fwhm,sig_fwhm,bis_span,sig_bis_span,contrast,sig_contrast,...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,HD189733,59437.538315,-2174.760250,0.323563,7477.005863,0.647126,-32.279851,0.647126,57.246647,0.004955,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,HD189733,59437.542268,-2177.035396,0.295833,7475.083070,0.591666,-31.580881,0.591666,57.238499,0.004531,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [65]:
normalization_requests = [
    {
        "columns": ["iccf_rv", "iccf_fwhm", "iccf_bis", "iccf_vspan", "iccf_wspan", 'iccf_contrast'],
        "methods": ["subtract_mean", "fractional_mean", "zscore"],
        "group_column": "observation_file",
        "add_group_stats": True,
        "stat_columns": ["iccf_rv", "iccf_fwhm", "iccf_bis", "iccf_vspan", "iccf_wspan", 'iccf_contrast'],
    },
    {
        "columns": ["s_mw", "ha", "na", "ca", "rhk"],
        "methods": ["subtract_mean", "fractional_mean", "zscore"],
        "group_column": "observation_file",
        "add_group_stats": True,
        "stat_columns": ["s_mw", "ha", "na", "ca", "rhk"],
    },
]

all_ccf_df = utils.add_normalizations_from_requests(
    all_ccf_df,
    requests=normalization_requests,
)


In [66]:
"""
sample_catalog_df = load_sample_catalog("sample_sweetcat.csv")

extract_spectroscopy_archives(
    download_root="downloads/dace_ccf_a",
    extract_root="downloads/dace_ccf_a_extracted",
)

ccf_df = build_merged_ccf_table_for_observation(
    observation_name="HD189733_esp19_3.rdb",
    extracted_root="downloads/dace_ccf_a_extracted",
    observation_root="observations/Best_RM",
    corrected_root="observations/Best_RM/linear_corrected_with_uncertainties",
    sample_catalog_df=sample_catalog_df,
)

ccf_df.head()
"""

'\nsample_catalog_df = load_sample_catalog("sample_sweetcat.csv")\n\nextract_spectroscopy_archives(\n    download_root="downloads/dace_ccf_a",\n    extract_root="downloads/dace_ccf_a_extracted",\n)\n\nccf_df = build_merged_ccf_table_for_observation(\n    observation_name="HD189733_esp19_3.rdb",\n    extracted_root="downloads/dace_ccf_a_extracted",\n    observation_root="observations/Best_RM",\n    corrected_root="observations/Best_RM/linear_corrected_with_uncertainties",\n    sample_catalog_df=sample_catalog_df,\n)\n\nccf_df.head()\n'

In [67]:
rm_df = all_ccf_df

In [68]:
def plot_corrected_observation(obs_name, target_column):
    obs_df = rm_df.loc[rm_df['observation_file'] == obs_name].sort_values('rjd')

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=obs_df['rjd'],
            y=obs_df[target_column],
            mode='markers+lines',
            marker=dict(size=7),
            name=target_column,
            hovertemplate='rjd=%{x}<br>true_vrad=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=f'Linear-trend corrected RV: {obs_name}',
        xaxis_title='rjd',
        yaxis_title=target_column,
        template='plotly_white',
        hovermode='closest',
    )

    fig.show()


obs_selector = widgets.Dropdown(
    options=sorted(rm_df['observation_file'].unique()),
    description='obs_name:',
)
interactive_plot = widgets.interactive_output(plot_corrected_observation, {'obs_name': obs_selector},)
#display(obs_selector, interactive_plot)

In [69]:
full_feature_columns = [
 'iccf_fwhm_mean',
 'iccf_fwhm_std',
 'iccf_bis_mean',
 'iccf_bis_std',
 'iccf_vspan_mean',
 'iccf_vspan_std',
 'iccf_wspan_mean',
 'iccf_wspan_std',
 'iccf_contrast_mean',
 'iccf_contrast_std',
 'iccf_rv_subtract_mean',
 'iccf_fwhm_subtract_mean',
 'iccf_bis_subtract_mean',
 'iccf_vspan_subtract_mean',
 'iccf_wspan_subtract_mean',
 'iccf_contrast_subtract_mean',
 'iccf_fwhm_fractional_mean',
 'iccf_bis_fractional_mean',
 'iccf_vspan_fractional_mean',
 'iccf_wspan_fractional_mean',
 'iccf_contrast_fractional_mean',
 'iccf_fwhm_zscore',
 'iccf_bis_zscore',
 'iccf_vspan_zscore',
 'iccf_wspan_zscore',
 'iccf_contrast_zscore',
 's_mw_mean',
 's_mw_std',
 'ha_mean',
 'ha_std',
 'ca_mean',
 'ca_std',
 's_mw_subtract_mean',
 'ha_subtract_mean',
 'ca_subtract_mean',
 's_mw_fractional_mean',
 'ha_fractional_mean',
 'ca_fractional_mean',
 's_mw_zscore',
 'ha_zscore',
 'ca_zscore',
]

subtract_mean_features = [
 'iccf_fwhm_subtract_mean',
 'iccf_bis_subtract_mean',
 'iccf_vspan_subtract_mean',
 'iccf_wspan_subtract_mean',
 'iccf_contrast_subtract_mean',]


subtract_fractional_mean_features = [ 'iccf_fwhm_fractional_mean',
 'iccf_bis_fractional_mean',
 'iccf_vspan_fractional_mean',
 'iccf_wspan_fractional_mean',
 'iccf_contrast_fractional_mean',]

zscore_features = [
 'iccf_fwhm_zscore',
 'iccf_bis_zscore',
 'iccf_vspan_zscore',
 'iccf_wspan_zscore',
 'iccf_contrast_zscore',]

In [70]:
residual_column = 'rv_residual_true_minus_iccf'
rm_df = rm_df.copy()
rm_df[residual_column] = (
    pd.to_numeric(rm_df['true_vrad2'], errors='coerce')
    - pd.to_numeric(rm_df['iccf_rv_subtract_mean'], errors='coerce')
)

residual_feature_columns = list(
    dict.fromkeys(
        [
            *subtract_mean_features,
            #*subtract_fractional_mean_features,
            #*zscore_features,
            #'s_mw_subtract_mean',
            #'ha_subtract_mean',
            #'ca_subtract_mean',
        ]
    )
)

correlation_rows = []
for obs_name, obs_df in rm_df.groupby('observation_file', sort=True):
    for feature_column in residual_feature_columns:
        if feature_column not in obs_df.columns:
            continue

        correlation_df = obs_df[[residual_column, feature_column]].apply(
            pd.to_numeric,
            errors='coerce',
        ).dropna()

        if len(correlation_df) < 2:
            correlation_value = float('nan')
        elif (
            correlation_df[residual_column].nunique() < 2
            or correlation_df[feature_column].nunique() < 2
        ):
            correlation_value = float('nan')
        else:
            correlation_value = correlation_df[residual_column].corr(
                correlation_df[feature_column],
                method='pearson',
            )

        correlation_rows.append(
            {
                'observation_file': obs_name,
                'feature_column': feature_column,
                'n_rows': len(correlation_df),
                'correlation': correlation_value,
                'abs_correlation': abs(correlation_value) if pd.notna(correlation_value) else float('nan'),
            }
        )

residual_correlations_long_df = pd.DataFrame(correlation_rows).sort_values(
    ['observation_file', 'abs_correlation'],
    ascending=[True, False],
).reset_index(drop=True)

residual_correlations_wide_df = (
    residual_correlations_long_df
    .pivot(index='observation_file', columns='feature_column', values='correlation')
    .sort_index()
)

display(residual_correlations_wide_df)
residual_correlations_long_df.head(20)


feature_column,iccf_bis_subtract_mean,iccf_contrast_subtract_mean,iccf_fwhm_subtract_mean,iccf_vspan_subtract_mean,iccf_wspan_subtract_mean
observation_file,,,,,
HD189733_esp19_3.rdb,0.846371,-0.039831,0.045906,0.819249,0.736242
HD189733_esp19_4.rdb,0.716194,0.053405,0.004605,0.451859,0.466991
HD189733_harps03_3.rdb,0.681728,0.008844,-0.115892,0.699140,0.560345
HD189733_harps03_4.rdb,0.762910,0.196611,-0.239079,0.826319,0.594662
HD189733_harps03_5.rdb,0.580187,-0.014003,0.059554,0.623760,0.491140
HD209458_esp19_1.rdb,0.400102,0.017491,-0.052239,0.307767,0.606242
HD209458_esp19_4.rdb,0.484448,0.034073,-0.052657,0.404531,0.518517
K2139_esp19_1.rdb,-0.090840,-0.355060,0.093685,-0.083260,0.129767
WASP106_esp19_1.rdb,0.306255,-0.070470,0.011429,0.248020,0.144548


,observation_file,feature_column,n_rows,correlation,abs_correlation
0,HD189733_esp19_3.rdb,iccf_bis_subtract_mean,41,0.846371,0.846371
1,HD189733_esp19_3.rdb,iccf_vspan_subtract_mean,41,0.819249,0.819249
2,HD189733_esp19_3.rdb,iccf_wspan_subtract_mean,41,0.736242,0.736242
3,HD189733_esp19_3.rdb,iccf_fwhm_subtract_mean,41,0.045906,0.045906
4,HD189733_esp19_3.rdb,iccf_contrast_subtract_mean,41,-0.039831,0.039831
5,HD189733_esp19_4.rdb,iccf_bis_subtract_mean,43,0.716194,0.716194
6,HD189733_esp19_4.rdb,iccf_wspan_subtract_mean,43,0.466991,0.466991
7,HD189733_esp19_4.rdb,iccf_vspan_subtract_mean,43,0.451859,0.451859
8,HD189733_esp19_4.rdb,iccf_contrast_subtract_mean,43,0.053405,0.053405
9,HD189733_esp19_4.rdb,iccf_fwhm_subtract_mean,43,0.004605,0.004605


In [74]:
def plot_residual_vs_feature(obs_name, feature_column):
    plot_df = rm_df.loc[
        rm_df['observation_file'] == obs_name,
        ['rjd', residual_column, feature_column],
    ].copy()

    plot_df[residual_column] = pd.to_numeric(plot_df[residual_column], errors='coerce')
    plot_df[feature_column] = pd.to_numeric(plot_df[feature_column], errors='coerce')
    plot_df = plot_df.dropna(subset=[residual_column, feature_column]).sort_values('rjd')

    if plot_df.empty:
        print(f'No rows available for {obs_name} and feature {feature_column}.')
        return

    if len(plot_df) < 2 or plot_df[residual_column].nunique() < 2 or plot_df[feature_column].nunique() < 2:
        correlation_value = float('nan')
    else:
        correlation_value = plot_df[residual_column].corr(plot_df[feature_column], method='pearson')

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=plot_df[feature_column],
            y=plot_df[residual_column],
            mode='markers',
            marker=dict(size=8, color=plot_df['rjd'], colorbar=dict(title='rjd')),
            customdata=plot_df[['rjd']],
            name='residual',
            hovertemplate='rjd=%{customdata[0]}<br>feature=%{x}<br>residual=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=(
            f'Residual vs feature: {obs_name}<br>'
            f'{feature_column} | corr={correlation_value:.3f} | n={len(plot_df)}'
        ),
        xaxis_title=feature_column,
        yaxis_title=f"{residual_column} = true_vrad2 - iccf_rv_subtract_mean",
        template='plotly_white',
        hovermode='closest',
    )
    fig.show()

    display(plot_df.head())


available_observations = sorted(rm_df['observation_file'].dropna().unique())
available_residual_features = list(residual_feature_columns)

selected_observation = available_observations[0]
selected_feature = available_residual_features[1]

print('Set `selected_observation` and `selected_feature`, then rerun this cell.')
print('Example observation:', selected_observation)
print('Example feature:', selected_feature)

plot_residual_vs_feature(selected_observation, selected_feature)


Set `selected_observation` and `selected_feature`, then rerun this cell.
Example observation: HD189733_esp19_3.rdb
Example feature: iccf_bis_subtract_mean


,rjd,rv_residual_true_minus_iccf,iccf_bis_subtract_mean
0,59437.538315,0.298789,-1.331155
1,59437.542268,0.284017,-0.631873
2,59437.546398,0.071242,-1.982753
3,59437.550744,0.253465,-1.228873
4,59437.554865,-0.229200,-1.961057


In [72]:
test_stars = ['K2139', 'WASP126', 'WASP166']
train_rm_df = rm_df.loc[~rm_df['star_from_file'].isin(test_stars)].copy()
test_rm_df = rm_df.loc[rm_df['star_from_file'].isin(test_stars)].copy()
test_label = ', '.join(test_stars)

display(
    pd.Series(
        {
            'n_train_rows': len(train_rm_df),
            'n_test_rows': len(test_rm_df),
            'n_test_observations': test_rm_df['observation_file'].nunique(),
        }
    ).to_frame('value')
)

,value
n_train_rows,913
n_test_rows,258
n_test_observations,3
